# CE Span Analysis: Evaluating Causal Chain Entity Detection

## Motivation

In causal relation extraction, some text spans serve as **intermediate nodes** in multi-step causal chains — they are simultaneously the **effect** of a preceding cause and the **cause** of a subsequent effect. We call these **CE spans** (Cause+Effect spans).

**Example from the test set:**
> *"researchers employ shared decisionmaking…* → **collaborations perform** → *coleadership throughout the research process* → *study progression"*

The bolded spans carry **dual causal roles** — they are annotated with both `cause` and `effect` labels at the same text offsets.

## Research Questions

1. How many CE spans exist in the test set?
2. How well do different models (Joint Multi-Task, Sequential, Llama 3, Qwen3) detect these dual-role spans?
3. Using the project's native **coverage-mode, overlap-based** evaluation, what are the precision, recall, and F1 scores for CE detection?

## Evaluation Protocol

We follow the exact evaluation logic from `src/analysis/causal_eval.py`:
- **Coverage mode**: a predicted span counts as TP if it *overlaps* any gold span of the same label (exact boundary match not required).
- **CE-level**: a CE span is **Both** if the model predicts at least one overlapping `cause` span AND at least one overlapping `effect` span. **Partial** if only one label. **Missed** if neither.

## 1. Setup & Imports

In [1]:
import json
import ast
import csv
import os
from typing import List, Tuple, Dict, Set
from collections import defaultdict
import pandas as pd

# Paths relative to project root
PROJECT_ROOT = os.path.abspath('..')
TEST_CSV = os.path.join(PROJECT_ROOT, 'datasets/expert_multi_task_data/test.csv')
DOCCANO_TEST = os.path.join(PROJECT_ROOT, 'datasets/expert_multi_task_data/doccano_test.jsonl')
JOINT_PRED = os.path.join(PROJECT_ROOT, 'datasets/expert_multi_task_data/bert-softmax__span_only__thr0.75.jsonl')
SEQ_PRED = os.path.join(PROJECT_ROOT, 'reviewer_extra_analysis/sequential_learning/predictions/sequential_predictions_doccano.csv')
LLAMA_PRED = os.path.join(PROJECT_ROOT, 'reviewer_extra_analysis/llm_evaluation/outputs/doccano/llama3_8b_test_doccano.csv')
QWEN_PRED = os.path.join(PROJECT_ROOT, 'reviewer_extra_analysis/llm_evaluation/outputs/doccano/qwen3_8b_test_doccano.csv')

print('Project root:', PROJECT_ROOT)
print('All paths resolved.')

Project root: /home/rnorouzini/JointLearning
All paths resolved.


## 2. Load Ground Truth & Detect CE Spans

A **CE span** is defined as a `(start_offset, end_offset)` that appears in the entity list **twice** — once with label `cause` and once with label `effect`.

In [2]:
def load_doccano_test(path: str) -> List[dict]:
    """Load doccano_test.jsonl."""
    samples = []
    with open(path, 'r') as f:
        for line in f:
            samples.append(json.loads(line.strip()))
    return samples

def find_ce_spans(samples: List[dict]) -> Dict[int, dict]:
    """
    Find all CE spans across samples.
    
    A CE span = same (start_offset, end_offset) appears with BOTH 
    'cause' and 'effect' labels in the entity list.
    
    Returns: dict mapping sample_id -> {
        'text': str,
        'ce_spans': {span_tuple: {'text': str, 'labels': set}},
        'all_entities': list,
    }
    """
    ce_data = {}
    for sample in samples:
        sid = sample['id']
        text = sample['text']
        entities = sample.get('entities', [])
        
        # Group entities by their span offsets
        span_labels = defaultdict(set)
        for e in entities:
            key = (e['start_offset'], e['end_offset'])
            span_labels[key].add(e['label'])
        
        # Find spans that have BOTH cause AND effect
        ce_spans = {}
        for span, labels in span_labels.items():
            if 'cause' in labels and 'effect' in labels:
                ce_spans[span] = {
                    'text': text[span[0]:span[1]],
                    'labels': labels,
                }
        
        if ce_spans:
            ce_data[sid] = {
                'text': text,
                'ce_spans': ce_spans,
                'all_entities': entities,
                'relations': sample.get('relations', []),
            }
    
    return ce_data

# Load
doccano_samples = load_doccano_test(DOCCANO_TEST)
ce_data = find_ce_spans(doccano_samples)

total_samples = len(doccano_samples)
total_ce_samples = len(ce_data)
total_ce_spans = sum(len(v['ce_spans']) for v in ce_data.values())

print(f'Total test samples: {total_samples}')
print(f'Samples with CE spans: {total_ce_samples} ({total_ce_samples/total_samples*100:.1f}%)')
print(f'Total CE spans: {total_ce_spans}')

Total test samples: 452
Samples with CE spans: 17 (3.8%)
Total CE spans: 20


## 3. Full Catalog of CE Spans in Test Set

In [3]:
# Build a flat list for display
ce_flat = []
for sid, data in sorted(ce_data.items()):
    for (s, e), info in data['ce_spans'].items():
        ce_flat.append({
            'Sample ID': sid,
            'Start': s,
            'End': e,
            'CE Span Text': info['text'],
            'Length (chars)': e - s,
        })

df_ce = pd.DataFrame(ce_flat)
df_ce.index = [f'CE#{i+1}' for i in range(len(df_ce))]
df_ce

,Sample ID,Start,End,CE Span Text,Length (chars)
CE#1,5757,38,96,understand patients' social support and contex...,58
CE#2,5925,131,150,increased awareness,19
CE#3,6306,5,27,collaborations perform,22
CE#4,6306,134,178,coleadership throughout the research process,44
CE#5,6310,123,164,power imbued in the researcher's position,41
CE#6,6330,82,147,consumers to want more money as a means to sec...,65
CE#7,6376,42,85,must involve negotiation among stakeholders,43
CE#8,6693,97,124,devaluation is not expected,27
CE#9,6738,56,112,ostracized participants experiencing processin...,56
CE#10,6738,276,316,decreasing executive processing capacity,40


### CE Span Length Distribution

In [4]:
print(f'Length stats (chars): min={df_ce["Length (chars)"].min()}, '
      f'mean={df_ce["Length (chars)"].mean():.1f}, '
      f'median={df_ce["Length (chars)"].median():.1f}, '
      f'max={df_ce["Length (chars)"].max()}')

Length stats (chars): min=19, mean=48.0, median=43.5, max=104


## 4. Evaluation Logic (replicating `causal_eval.py` coverage mode)

We implement the exact overlap-based matching from `src/analysis/causal_eval.py`:
- Two spans **overlap** if `max(a[0], b[0]) < min(a[1], b[1])`
- A predicted span of label X counts as a **TP** for a CE span if it overlaps the CE span and the CE span requires label X
- A CE span is **Both** if it gets overlapping predictions for BOTH cause and effect
- **Partial** if only one label
- **Missed** if neither

In [5]:
Span = Tuple[int, int]

def intervals_overlap(a: Span, b: Span) -> bool:
    """Exact copy from causal_eval.py."""
    return max(a[0], b[0]) < min(a[1], b[1])

def evaluate_ce_spans(
    ce_data: Dict[int, dict],
    model_preds: Dict[int, Dict[str, List[Span]]],
) -> dict:
    """
    Evaluate CE span detection using coverage-mode overlap matching.
    
    Args:
        ce_data: output of find_ce_spans() — ground truth CE spans
        model_preds: dict sample_id -> {'cause': [spans], 'effect': [spans]}
    
    Returns:
        dict with aggregated CE metrics
    """
    both = 0       # CE span got BOTH cause and effect overlapping preds
    partial = 0    # CE span got only ONE of the two
    missed = 0     # CE span got NEITHER
    
    tp_cause = 0   # CE span's cause label was matched
    fn_cause = 0   # CE span's cause label was missed
    tp_effect = 0
    fn_effect = 0
    
    per_sample = []
    
    for sid in sorted(ce_data.keys()):
        gt = ce_data[sid]
        pred = model_preds.get(sid, {'cause': [], 'effect': []})
        
        for ce_span in gt['ce_spans']:
            has_cause = any(intervals_overlap(p, ce_span) for p in pred['cause'])
            has_effect = any(intervals_overlap(p, ce_span) for p in pred['effect'])
            
            if has_cause:
                tp_cause += 1
            else:
                fn_cause += 1
            
            if has_effect:
                tp_effect += 1
            else:
                fn_effect += 1
            
            if has_cause and has_effect:
                both += 1
                status = 'Both'
            elif has_cause or has_effect:
                partial += 1
                status = 'Partial'
            else:
                missed += 1
                status = 'Missed'
            
            per_sample.append({
                'sample_id': sid,
                'ce_span': ce_span,
                'ce_text': gt['ce_spans'][ce_span]['text'],
                'has_cause': has_cause,
                'has_effect': has_effect,
                'status': status,
            })
    
    total = both + partial + missed
    p_chain = both / (both + partial) if (both + partial) > 0 else 0.0
    r_chain = both / total if total > 0 else 0.0
    f1_chain = 2 * p_chain * r_chain / (p_chain + r_chain) if (p_chain + r_chain) > 0 else 0.0
    
    p_cause = tp_cause / total if total > 0 else 0.0  # simplified: FP not counted per-CE-span
    r_cause = tp_cause / (tp_cause + fn_cause) if (tp_cause + fn_cause) > 0 else 0.0
    f1_cause = 2 * p_cause * r_cause / (p_cause + r_cause) if (p_cause + r_cause) > 0 else 0.0
    
    p_effect = tp_effect / total if total > 0 else 0.0
    r_effect = tp_effect / (tp_effect + fn_effect) if (tp_effect + fn_effect) > 0 else 0.0
    f1_effect = 2 * p_effect * r_effect / (p_effect + r_effect) if (p_effect + r_effect) > 0 else 0.0
    
    return {
        'total_ce_spans': total,
        'both': both, 'partial': partial, 'missed': missed,
        'p_chain': p_chain, 'r_chain': r_chain, 'f1_chain': f1_chain,
        'tp_cause': tp_cause, 'fn_cause': fn_cause,
        'tp_effect': tp_effect, 'fn_effect': fn_effect,
        'p_cause': p_cause, 'r_cause': r_cause, 'f1_cause': f1_cause,
        'p_effect': p_effect, 'r_effect': r_effect, 'f1_effect': f1_effect,
        'per_sample': per_sample,
    }

print('Evaluation functions defined.')

Evaluation functions defined.


## 5. Load All Model Predictions

In [6]:
def get_span_lists(entities: list) -> Dict[str, List[Span]]:
    """Extract cause and effect span lists from entity list."""
    cause_spans = []
    effect_spans = []
    for e in entities:
        lbl = e['label'].lower()
        span = (e['start_offset'], e['end_offset'])
        if lbl == 'cause':
            cause_spans.append(span)
        elif lbl == 'effect':
            effect_spans.append(span)
    return {'cause': cause_spans, 'effect': effect_spans}


# Build ID -> index mapping from test.csv
id_to_idx = {}
with open(TEST_CSV, 'r') as f:
    reader = csv.reader(f, quotechar='"')
    next(reader)  # skip header
    for i, row in enumerate(reader):
        id_to_idx[int(row[0])] = i

ce_ids = sorted(ce_data.keys())

# ---- Joint (Multi-Task) Model ----
with open(JOINT_PRED, 'r') as f:
    joint_all = [json.loads(line.strip()) for line in f]
joint_preds = {sid: get_span_lists(joint_all[id_to_idx[sid]]['entities']) for sid in ce_ids}

# ---- Sequential Model ----
with open(SEQ_PRED, 'r') as f:
    reader = csv.reader(f, quotechar='"')
    next(reader)
    seq_rows = list(reader)
seq_preds = {}
for sid in ce_ids:
    idx = id_to_idx[sid]
    entities = ast.literal_eval(seq_rows[idx][2])
    seq_preds[sid] = get_span_lists(entities)

# ---- Llama 3 8B ----
with open(LLAMA_PRED, 'r') as f:
    reader = csv.reader(f, quotechar='"')
    next(reader)
    llama_rows = list(reader)
llama_preds = {}
for sid in ce_ids:
    idx = id_to_idx[sid]
    entities = ast.literal_eval(llama_rows[idx][2])
    llama_preds[sid] = get_span_lists(entities)

# ---- Qwen3 8B ----
with open(QWEN_PRED, 'r') as f:
    reader = csv.reader(f, quotechar='"')
    next(reader)
    qwen_rows = list(reader)
qwen_preds = {}
for sid in ce_ids:
    idx = id_to_idx[sid]
    entities = ast.literal_eval(qwen_rows[idx][2])
    qwen_preds[sid] = get_span_lists(entities)

print(f'Loaded predictions for {len(ce_ids)} CE samples across 4 models.')

Loaded predictions for 17 CE samples across 4 models.


## 6. Run CE Evaluation on All Models

In [7]:
models = {
    'Joint (Multi-Task)': joint_preds,
    'Sequential': seq_preds,
    'Llama 3 8B': llama_preds,
    'Qwen3 8B': qwen_preds,
}

all_results = {}
for name, preds in models.items():
    all_results[name] = evaluate_ce_spans(ce_data, preds)
    print(f'{name:<22} Both={all_results[name]["both"]:>2}  '
          f'Partial={all_results[name]["partial"]:>2}  Missed={all_results[name]["missed"]:>2}  '
          f'F1(chain)={all_results[name]["f1_chain"]:.4f}')

Joint (Multi-Task)     Both= 7  Partial=11  Missed= 2  F1(chain)=0.3684
Sequential             Both= 9  Partial=11  Missed= 0  F1(chain)=0.4500
Llama 3 8B             Both= 8  Partial= 6  Missed= 6  F1(chain)=0.4706
Qwen3 8B               Both= 9  Partial=10  Missed= 1  F1(chain)=0.4615


## 7. Summary Results Table

### 7a. CE-Level Detection (Both / Partial / Missed)

In [8]:
rows_chain = []
for name, r in all_results.items():
    rows_chain.append({
        'Model': name,
        'Both': r['both'],
        'Partial': r['partial'],
        'Missed': r['missed'],
        'P(both)': f"{r['p_chain']:.4f}",
        'R(both)': f"{r['r_chain']:.4f}",
        'F1(both)': f"{r['f1_chain']:.4f}",
    })

df_chain = pd.DataFrame(rows_chain)
df_chain

,Model,Both,Partial,Missed,P(both),R(both),F1(both)
0,Joint (Multi-Task),7,11,2,0.3889,0.3500,0.3684
1,Sequential,9,11,0,0.4500,0.4500,0.4500
2,Llama 3 8B,8,6,6,0.5714,0.4000,0.4706
3,Qwen3 8B,9,10,1,0.4737,0.4500,0.4615


### 7b. Per-Label Metrics on CE Spans

In [9]:
rows_label = []
for name, r in all_results.items():
    for lbl in ['cause', 'effect']:
        rows_label.append({
            'Model': name,
            'Label': lbl,
            'TP': r[f'tp_{lbl}'],
            'FN': r[f'fn_{lbl}'],
            'Precision': f"{r[f'p_{lbl}']:.4f}",
            'Recall': f"{r[f'r_{lbl}']:.4f}",
            'F1': f"{r[f'f1_{lbl}']:.4f}",
        })

df_labels = pd.DataFrame(rows_label)
df_labels

,Model,Label,TP,FN,Precision,Recall,F1
0,Joint (Multi-Task),cause,12,8,0.6000,0.6000,0.6000
1,Joint (Multi-Task),effect,13,7,0.6500,0.6500,0.6500
2,Sequential,cause,14,6,0.7000,0.7000,0.7000
3,Sequential,effect,15,5,0.7500,0.7500,0.7500
4,Llama 3 8B,cause,14,6,0.7000,0.7000,0.7000
5,Llama 3 8B,effect,8,12,0.4000,0.4000,0.4000
6,Qwen3 8B,cause,16,4,0.8000,0.8000,0.8000
7,Qwen3 8B,effect,12,8,0.6000,0.6000,0.6000


## 8. Per-Sample Breakdown

Shows exactly which CE spans each model detected, partially detected, or missed.

In [10]:
# Build a comprehensive per-sample table
per_sample_rows = []
for i, (sid, gt) in enumerate(sorted(ce_data.items())):
    for j, (ce_span, info) in enumerate(gt['ce_spans'].items()):
        row = {
            'CE#': f'CE#{sum(len(ce_data[s]["ce_spans"]) for s in sorted(ce_data.keys()) if s < sid) + j + 1}',
            'Sample': sid,
            'CE Span': info['text'][:70] + ('...' if len(info['text']) > 70 else ''),
        }
        for model_name in ['Joint (Multi-Task)', 'Sequential', 'Llama 3 8B', 'Qwen3 8B']:
            per_sample = all_results[model_name]['per_sample']
            entry = next((e for e in per_sample if e['sample_id'] == sid and e['ce_span'] == ce_span), None)
            if entry:
                row[model_name] = entry['status']
        per_sample_rows.append(row)

df_per_sample = pd.DataFrame(per_sample_rows)
# Color-code: Both=green, Partial=yellow, Missed=red
def color_status(val):
    if val == 'Both':
        return 'background-color: #c6efce; color: #006100'
    elif val == 'Partial':
        return 'background-color: #ffeb9c; color: #9c5700'
    elif val == 'Missed':
        return 'background-color: #ffc7ce; color: #9c0006'
    return ''

df_per_sample.style.map(color_status, subset=['Joint (Multi-Task)', 'Sequential', 'Llama 3 8B', 'Qwen3 8B'])

,CE#,Sample,CE Span,Joint (Multi-Task),Sequential,Llama 3 8B,Qwen3 8B
0,CE#1,5757,understand patients' social support and contextual factors,Missed,Both,Both,Both
1,CE#2,5925,increased awareness,Partial,Partial,Partial,Partial
2,CE#3,6306,collaborations perform,Missed,Partial,Missed,Missed
3,CE#4,6306,coleadership throughout the research process,Partial,Partial,Both,Both
4,CE#5,6310,power imbued in the researcher's position,Both,Both,Both,Both
5,CE#6,6330,consumers to want more money as a means to secure control in life,Both,Both,Both,Both
6,CE#7,6376,must involve negotiation among stakeholders,Partial,Partial,Both,Both
7,CE#8,6693,devaluation is not expected,Both,Partial,Partial,Both
8,CE#9,6738,ostracized participants experiencing processing deficits,Partial,Both,Missed,Partial
9,CE#10,6738,decreasing executive processing capacity,Partial,Both,Missed,Partial


## 9. Worked Example: Step-by-Step Evaluation

Let's trace through the evaluation for one specific CE sample to make the logic concrete.

In [11]:
# Pick sample 6306 (has 2 CE spans)
EXAMPLE_SID = 6306
gt = ce_data[EXAMPLE_SID]

print('='*70)
print(f'WORKED EXAMPLE: Sample {EXAMPLE_SID}')
print('='*70)
print(f'\nFULL TEXT:\n  {gt["text"]}\n')

print('GOLD ENTITIES:')
for e in sorted(gt['all_entities'], key=lambda x: x['start_offset']):
    is_ce = (e['start_offset'], e['end_offset']) in gt['ce_spans']
    marker = ' <<< CE SPAN' if is_ce else ''
    print(f'  [{e["start_offset"]:>3}:{e["end_offset"]:>3}] label={e["label"]:<7}  '
          f'"{gt["text"][e["start_offset"]:e["end_offset"]]}"{marker}')

print(f'\nCE SPANS: {len(gt["ce_spans"])}')
for span, info in gt['ce_spans'].items():
    print(f'  [{span[0]}:{span[1]}] "{info["text"]}"  labels: {info["labels"]}')

WORKED EXAMPLE: Sample 6306

FULL TEXT:
  such collaborations perform best when researchers employ shared decisionmaking processes with communities of study, thereby promoting coleadership throughout the research process that facilitates both study progression and community agency (sprague et al., 2019).;;

GOLD ENTITIES:
  [  5: 27] label=effect   "collaborations perform" <<< CE SPAN
  [  5: 27] label=cause    "collaborations perform" <<< CE SPAN
  [ 38:114] label=cause    "researchers employ shared decisionmaking processes with communities of study"
  [134:178] label=effect   "coleadership throughout the research process" <<< CE SPAN
  [134:178] label=cause    "coleadership throughout the research process" <<< CE SPAN
  [201:218] label=effect   "study progression"
  [223:239] label=effect   "community agency"

CE SPANS: 2
  [5:27] "collaborations perform"  labels: {'cause', 'effect'}
  [134:178] "coleadership throughout the research process"  labels: {'cause', 'effect'}


In [12]:
# Show Joint model predictions for this sample
idx = id_to_idx[EXAMPLE_SID]
joint_entities = joint_all[idx]['entities']

print('JOINT MODEL PREDICTED ENTITIES:')
for e in sorted(joint_entities, key=lambda x: x['start_offset']):
    print(f'  [{e["start_offset"]:>3}:{e["end_offset"]:>3}] label={e["label"]:<7}  '
          f'"{gt["text"][e["start_offset"]:e["end_offset"]]}"')

JOINT MODEL PREDICTED ENTITIES:
  [ 38:114] label=cause    "researchers employ shared decisionmaking processes with communities of study"
  [134:178] label=effect   "coleadership throughout the research process"
  [184:239] label=effect   "facilitates both study progression and community agency"


In [13]:
# Step-by-step overlap check
model_name = 'Joint (Multi-Task)'
pred = joint_preds[EXAMPLE_SID]

print(f'Predicted cause spans: {pred["cause"]}')
print(f'Predicted effect spans: {pred["effect"]}')
print()

for ce_span, info in gt['ce_spans'].items():
    print(f'CE Span [{ce_span[0]}:{ce_span[1]}] "{info["text"]}"')
    print(f'  Gold labels: {info["labels"]}')
    
    cause_overlaps = [p for p in pred['cause'] if intervals_overlap(p, ce_span)]
    effect_overlaps = [p for p in pred['effect'] if intervals_overlap(p, ce_span)]
    
    print(f'  Cause check: {"✓" if cause_overlaps else "✗"} overlapping preds: {cause_overlaps if cause_overlaps else "none"}')
    print(f'  Effect check: {"✓" if effect_overlaps else "✗"} overlapping preds: {effect_overlaps if effect_overlaps else "none"}')
    
    has_c = len(cause_overlaps) > 0
    has_e = len(effect_overlaps) > 0
    
    if has_c and has_e:
        verdict = 'Both ✓✓'
    elif has_c or has_e:
        which = 'CAUSE only' if has_c else 'EFFECT only'
        verdict = f'Partial ({which})'
    else:
        verdict = 'Missed ✗✗'
    
    print(f'  >>> VERDICT: {verdict}')
    print()

Predicted cause spans: [(38, 114)]
Predicted effect spans: [(184, 239), (134, 178)]

CE Span [5:27] "collaborations perform"
  Gold labels: {'cause', 'effect'}
  Cause check: ✗ overlapping preds: none
  Effect check: ✗ overlapping preds: none
  >>> VERDICT: Missed ✗✗

CE Span [134:178] "coleadership throughout the research process"
  Gold labels: {'cause', 'effect'}
  Cause check: ✗ overlapping preds: none
  Effect check: ✓ overlapping preds: [(134, 178)]
  >>> VERDICT: Partial (EFFECT only)



## 10. Findings & Interpretation

### Key Observations

1. **No model fully masters CE detection.** The best CE-level F1 is 0.47 (Llama 3 8B), meaning fewer than half of CE spans are fully recognized with both causal roles.

2. **Most CE spans are at least partially detected.** Only 0–6 out of 20 CE spans are completely missed by any model. The failure mode is incomplete labeling — models detect a span overlapping the CE entity but assign only one of the two required labels.

3. **The Sequential model misses zero CE spans** (Missed=0), but has the lowest precision (P(both)=0.45) — it finds everything but often fails to assign the second label.

4. **Llama 3 8B has the highest precision on CE spans** (P(both)=0.57) — when it does detect a CE span, it's more likely to get both labels right. However, it also has the most completely missed spans (6).

5. **Effect labels are harder than cause labels on CE spans for LLMs.** Llama 3 shows a striking asymmetry (cause recall 0.70 vs. effect recall 0.40 on CE spans), while BERT-based models maintain more balanced recall.

### Implications

The dual-role nature of CE spans fundamentally challenges models that treat cause and effect as mutually exclusive categories. Even with lenient overlap-based evaluation, all models show significant degradation on CE spans compared to their overall Task 2 performance. Future work could explore multi-label classification heads or reformulating the task at the token level to allow joint cause+effect assignment to the same span.